# 직접 실행: CN7/RG3 DNN만 학습

완료된 AE·SVM·RF·GaussianNB 7개 결과는 포함되어 있다. 이 노트북은 해당 모델을 재학습하지 않고 DNN 두 제품만 실행한다. 가이드의 구조·최대 100 epoch·상위10% 추가·약90% 사용 조건을 유지한다. 각 제품은 22회 반복한다.

1. Colab에 이 파일을 업로드하고 런타임을 선택한다. GPU는 선택 사항이며 이 소형 모델에서 속도 향상을 보장하지 않는다.
2. 1번 셀의 DATA_FOLDER를 CSV가 있는 Drive 폴더로 수정한다.
3. 1→6번 순서로 실행한다. 4번 CN7, 5번 RG3가 오래 걸리는 학습 셀이다.
4. 결과는 MyDrive/molding_dnn_manual_results에 자동 저장한다.

새 실행은 DNN 1회차부터 시작한다. 다른 환경으로 이전한 중간 가중치의 완전한 재현을 보장하지 않으므로 기존 로컬 중간 모델을 자동으로 이어 쓰지 않는다. 이 노트북에서 한 제품의 학습을 완료하면 다음 실행에서 그 제품은 건너뛴다. 중간 체크포인트는 보관하지만 미완료 제품의 자동 이어학습 기능은 아니다.

원문에 대한 호환 수정과 평가 조건 차이는 기존 재현 README를 따른다. Colab과 로컬의 버전·장치 차이로 수치가 달라질 수 있다. AE와 준지도 모델은 평가 표본 및 전처리가 다르므로 점수를 직접 비교하지 않는다.

## 1. Drive 연결·데이터 폴더 지정

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
from pathlib import Path

# CSV 파일 하나가 아니라, 네 개의 moldset CSV가 들어 있는 폴더 경로다.
DATA_FOLDER = '/content/drive/MyDrive/사출 공정 불량 예측 및 검사 기준 분석/dataset'
OUTPUT_FOLDER = '/content/drive/MyDrive/molding_dnn_manual_results'

required=['moldset_labeled_cn7.csv','moldset_labeled_rg3.csv',
          'moldset_unlabeled_cn7.csv','moldset_unlabeled_rg3.csv']
missing=[name for name in required if not (Path(DATA_FOLDER)/name).is_file()]
assert not missing, f'폴더 경로를 확인하세요. 찾지 못한 파일: {missing}'
os.environ['MOLDING_DATA_DIR']=DATA_FOLDER
os.environ['REFERENCE_RESULTS']=OUTPUT_FOLDER
print('데이터 파일 4개 확인 완료')
print('결과 저장 폴더:',OUTPUT_FOLDER)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
데이터 파일 4개 확인 완료
결과 저장 폴더: /content/drive/MyDrive/molding_dnn_manual_results


## 2. 실행 환경·완료된 모델 결과 불러오기

In [3]:
import os, json, time, hashlib
from pathlib import Path
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, average_precision_score, roc_auc_score)
from IPython.display import display
tf.config.threading.set_intra_op_parallelism_threads(2)
tf.config.threading.set_inter_op_parallelism_threads(2)
tf.keras.utils.set_random_seed(42)
DATA_DIR=Path(os.environ.get('MOLDING_DATA_DIR', '/content/drive/MyDrive/dataset'))
RESULT_DIR=Path(os.environ.get('REFERENCE_RESULTS', 'results'))
RESULT_DIR.mkdir(parents=True, exist_ok=True)
def read(name):
    d=pd.read_csv(DATA_DIR/name)
    return d.loc[:, ~d.columns.str.startswith('Unnamed:')]
rows=[]
def metrics(y, pred, score):
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return dict(TN=int(tn),FP=int(fp),FN=int(fn),TP=int(tp),
        accuracy=accuracy_score(y,pred),precision=precision_score(y,pred,zero_division=0),
        recall=recall_score(y,pred,zero_division=0),F1=f1_score(y,pred,zero_division=0),
        AP=average_precision_score(y,score),ROC_AUC_score=roc_auc_score(y,score),
        guide_ROC_AUC_hard=roc_auc_score(y,pred))
def record(row):
    rows.append(row)
    pd.DataFrame(rows).to_csv(RESULT_DIR/'metrics.csv',index=False)
    display(pd.DataFrame([row]))
def progress(message):
    with (RESULT_DIR/'progress.txt').open('a',encoding='utf-8') as f:
        f.write(time.strftime('%H:%M:%S')+' '+message+'\n')
    print(message,flush=True)
manifest=[]
for name in ['moldset_labeled_cn7.csv',
             'moldset_labeled_rg3.csv','moldset_unlabeled_cn7.csv','moldset_unlabeled_rg3.csv']:
    d=read(name)
    manifest.append(dict(file=name,rows=len(d),columns=len(d.columns),
                         sha256=hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest()))
display(pd.DataFrame(manifest))
pd.DataFrame(manifest).to_csv(RESULT_DIR/'input_manifest.csv',index=False)
print('TensorFlow',tf.__version__)

BASE_RESULTS=json.loads('[{"track": "guide_AE", "product": "CN7", "model": "DenoisingAE", "epochs": 30.0, "threshold": 0.0769900531, "TN": 2695, "FP": 2, "FN": 0, "TP": 39, "accuracy": 0.9992690058, "precision": 0.9512195122, "recall": 1.0, "F1": 0.975, "AP": 1.0, "ROC_AUC_score": 1.0, "guide_ROC_AUC_hard": 0.9996292176, "iterations": null, "trained": null, "assigned_not_refit": null}, {"track": "guide_semi", "product": "CN7", "model": "SVM", "epochs": null, "threshold": null, "TN": 359, "FP": 0, "FN": 5, "TP": 0, "accuracy": 0.9862637363, "precision": 0.0, "recall": 0.0, "F1": 0.0, "AP": 0.4725816518, "ROC_AUC_score": 0.9286908078, "guide_ROC_AUC_hard": 0.5, "iterations": 22.0, "trained": 32227.0, "assigned_not_refit": 385.0}, {"track": "guide_semi", "product": "CN7", "model": "RandomForest", "epochs": null, "threshold": null, "TN": 357, "FP": 2, "FN": 3, "TP": 2, "accuracy": 0.9862637363, "precision": 0.5, "recall": 0.4, "F1": 0.4444444444, "AP": 0.4756462585, "ROC_AUC_score": 0.9509749304, "guide_ROC_AUC_hard": 0.6972144847, "iterations": 22.0, "trained": 32227.0, "assigned_not_refit": 385.0}, {"track": "guide_semi", "product": "CN7", "model": "GaussianNB", "epochs": null, "threshold": null, "TN": 294, "FP": 65, "FN": 1, "TP": 4, "accuracy": 0.8186813187, "precision": 0.0579710145, "recall": 0.8, "F1": 0.1081081081, "AP": 0.4412087912, "ROC_AUC_score": 0.8423398329, "guide_ROC_AUC_hard": 0.8094707521, "iterations": 22.0, "trained": 32227.0, "assigned_not_refit": 385.0}, {"track": "guide_semi", "product": "RG3", "model": "SVM", "epochs": null, "threshold": null, "TN": 347, "FP": 0, "FN": 8, "TP": 0, "accuracy": 0.9774647887, "precision": 0.0, "recall": 0.0, "F1": 0.0, "AP": 0.0280496686, "ROC_AUC_score": 0.538184438, "guide_ROC_AUC_hard": 0.5, "iterations": 22.0, "trained": 32831.0, "assigned_not_refit": 393.0}, {"track": "guide_semi", "product": "RG3", "model": "RandomForest", "epochs": null, "threshold": null, "TN": 341, "FP": 6, "FN": 8, "TP": 0, "accuracy": 0.9605633803, "precision": 0.0, "recall": 0.0, "F1": 0.0, "AP": 0.02968101, "ROC_AUC_score": 0.5385446686, "guide_ROC_AUC_hard": 0.4913544669, "iterations": 22.0, "trained": 32831.0, "assigned_not_refit": 393.0}, {"track": "guide_semi", "product": "RG3", "model": "GaussianNB", "epochs": null, "threshold": null, "TN": 10, "FP": 337, "FN": 0, "TP": 8, "accuracy": 0.0507042254, "precision": 0.0231884058, "recall": 1.0, "F1": 0.045325779, "AP": 0.0208181326, "ROC_AUC_score": 0.4153458213, "guide_ROC_AUC_hard": 0.5144092219, "iterations": 22.0, "trained": 32831.0, "assigned_not_refit": 393.0}]')
# 이 수동 실행 폴더에서 완료한 DNN만 복원한다.
if (RESULT_DIR/'metrics.csv').exists():
    prior=pd.read_csv(RESULT_DIR/'metrics.csv')
    completed=prior.loc[(prior.model=='DNN') & (prior.iterations==22)].to_dict('records')
else: completed=[]
rows=BASE_RESULTS+completed
pd.DataFrame(rows).to_csv(RESULT_DIR/'metrics.csv',index=False)
import sys, sklearn
(RESULT_DIR/'environment.json').write_text(json.dumps({
    'python':sys.version,'tensorflow':tf.__version__,'numpy':np.__version__,
    'pandas':pd.__version__,'sklearn':sklearn.__version__,
    'gpu':[d.name for d in tf.config.list_physical_devices('GPU')]
},ensure_ascii=False,indent=2),encoding='utf-8')
print('GPU:',tf.config.list_physical_devices('GPU'))
print('이미 완료한 DNN 제품:',[r['product'] for r in completed])


,file,rows,columns,sha256
0,moldset_labeled_cn7.csv,1211,25,f870b0c259297f20d503f2e2739966553e5db5ef1deb0c...
1,moldset_labeled_rg3.csv,1182,25,14aab21476eea02c823ccb00f82800040280d805dbce5a...
2,moldset_unlabeled_cn7.csv,35239,24,110a2d408d6cb8002caf26f50c06d06e42998e067e7087...
3,moldset_unlabeled_rg3.csv,35941,24,ef9b30014ce4d6671a72f5a4c3f286802b71ff08d52f9d...


TensorFlow 2.20.0
GPU: []
이미 완료한 DNN 제품: []


## 3. 데이터 분리·DNN 함수 정의 — 여기서는 아직 학습하지 않는다

In [4]:
datasets={}
for product in ['cn7','rg3']:
    d=read(f'moldset_labeled_{product}.csv')
    u=read(f'moldset_unlabeled_{product}.csv')
    y=d.PassOrFail.astype(int).to_numpy()
    x=d.drop(columns='PassOrFail')
    assert list(x.columns)==list(u.columns)
    assert np.isfinite(x.to_numpy()).all() and np.isfinite(u.to_numpy()).all()
    tr,te=next(StratifiedShuffleSplit(n_splits=1,test_size=.3,random_state=42).split(x,y))
    datasets[product]=(x.iloc[tr].to_numpy(),y[tr],x.iloc[te].to_numpy(),y[te],u.to_numpy())
    print(product, '학습',len(tr),'평가',len(te),'평가 불량',int(y[te].sum()),'미라벨',len(u))
def select_pseudo(u,score):
    confidence=np.maximum(score,1-score)
    order=pd.DataFrame({'confidence':confidence}).sort_values('confidence',ascending=False).index.to_numpy()
    k=int(len(u)*.1)
    assert k>0
    return order[:k],order[k:]

def run_dnn(product):
    if any(r.get('model')=='DNN' and r.get('product')==product.upper() for r in rows):
        print(product,'이미 완료됨: 저장된 결과 사용')
        return
    tf.keras.backend.clear_session();tf.keras.utils.set_random_seed(42)
    x,y,xt,yt,u=datasets[product]
    x=x.astype('float32');u=u.astype('float32');xt=xt.astype('float32');y=y.copy()
    model=tf.keras.Sequential([tf.keras.layers.Input(shape=(24,)),
        tf.keras.layers.Dense(32,activation='relu'),tf.keras.layers.Dense(64,activation='relu'),
        tf.keras.layers.Dropout(.25),tf.keras.layers.Dense(32,activation='relu'),
        tf.keras.layers.Dropout(.2),tf.keras.layers.Dense(16,activation='relu'),
        tf.keras.layers.Dense(1,activation='sigmoid')])
    model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy',
        tf.keras.metrics.Precision(name='precision'),tf.keras.metrics.Recall(name='recall')])
    limit=int(len(u)*.1);iteration=0;logs=[]
    while len(u)>=limit:
        iteration+=1;vp=0;vr=0;cnt=0;trained=len(y)
        for epoch in range(100):
            h=model.fit(x,y,epochs=1,validation_split=.3,verbose=0).history
            if cnt>=10: break
            if h['val_precision'][0]>=vp and h['val_recall'][0]>=vr:
                vp=h['val_precision'][0];vr=h['val_recall'][0];cnt=0
            else: cnt+=1
            if (epoch+1)%20==0: progress(f'{product} DNN 반복 {iteration}, epoch {epoch+1}')
        score=model.predict(xt,verbose=0).ravel();pred=(score>=.5).astype(int)
        logs.append(dict(iteration=iteration,epochs=epoch+1,trained=trained,**metrics(yt,pred,score)))
        us=model.predict(u,verbose=0).ravel();chosen,left=select_pseudo(u,us)
        pseudo=(model.predict(u[chosen],verbose=0).ravel()>=.5).astype(int)
        x=np.concatenate([x,u[chosen]]);y=np.concatenate([y,pseudo]);u=u[left]
        pd.DataFrame(logs).to_csv(RESULT_DIR/f'{product}_DNN_iterations.csv',index=False)
        # 장시간 실행의 진행 보존. 학습 설정 및 종료 규칙은 바꾸지 않는다.
        model.save(RESULT_DIR/f'{product}_dnn_checkpoint.keras')
        np.savez_compressed(RESULT_DIR/f'{product}_dnn_checkpoint_arrays.npz',
                            x=x,y=y,u=u,iteration=iteration)
        progress(f'{product} DNN 반복 {iteration} 완료: 남은 미라벨 {len(u)}')
    model.save(RESULT_DIR/f'{product}_dnn.keras')
    record(dict(track='guide_semi',product=product.upper(),model='DNN',iterations=iteration,
                trained=trained,assigned_not_refit=len(y)-trained,**metrics(yt,pred,score)))


cn7 학습 847 평가 364 평가 불량 5 미라벨 35239
rg3 학습 827 평가 355 평가 불량 8 미라벨 35941


## 4. CN7 학습 — 완료될 때까지 기다리기

로그의 `반복 22 완료`와 결과 표가 나오고 셀 실행 표시가 멈추면 완료다. epoch 100은 한 회차의 끝이며 전체 완료가 아니다.

In [5]:
run_dnn('cn7')

cn7 DNN 반복 1, epoch 20
cn7 DNN 반복 1, epoch 40
cn7 DNN 반복 1, epoch 60
cn7 DNN 반복 1, epoch 80
cn7 DNN 반복 1, epoch 100
cn7 DNN 반복 1 완료: 남은 미라벨 31716
cn7 DNN 반복 2, epoch 20
cn7 DNN 반복 2, epoch 40
cn7 DNN 반복 2, epoch 60
cn7 DNN 반복 2, epoch 80
cn7 DNN 반복 2, epoch 100
cn7 DNN 반복 2 완료: 남은 미라벨 28545
cn7 DNN 반복 3, epoch 20
cn7 DNN 반복 3, epoch 40
cn7 DNN 반복 3, epoch 60
cn7 DNN 반복 3, epoch 80
cn7 DNN 반복 3, epoch 100
cn7 DNN 반복 3 완료: 남은 미라벨 25691
cn7 DNN 반복 4, epoch 20
cn7 DNN 반복 4, epoch 40
cn7 DNN 반복 4, epoch 60
cn7 DNN 반복 4, epoch 80
cn7 DNN 반복 4, epoch 100
cn7 DNN 반복 4 완료: 남은 미라벨 23122
cn7 DNN 반복 5, epoch 20
cn7 DNN 반복 5, epoch 40
cn7 DNN 반복 5, epoch 60
cn7 DNN 반복 5, epoch 80
cn7 DNN 반복 5, epoch 100
cn7 DNN 반복 5 완료: 남은 미라벨 20810
cn7 DNN 반복 6, epoch 20
cn7 DNN 반복 6, epoch 40
cn7 DNN 반복 6, epoch 60
cn7 DNN 반복 6, epoch 80
cn7 DNN 반복 6, epoch 100
cn7 DNN 반복 6 완료: 남은 미라벨 18729
cn7 DNN 반복 7, epoch 20
cn7 DNN 반복 7, epoch 40
cn7 DNN 반복 7, epoch 60
cn7 DNN 반복 7, epoch 80
cn7 DNN 반복 7, epoch 100
cn7 DNN 

,track,product,model,iterations,trained,assigned_not_refit,TN,FP,FN,TP,accuracy,precision,recall,F1,AP,ROC_AUC_score,guide_ROC_AUC_hard
0,guide_semi,CN7,DNN,22,32227,385,357,2,3,2,0.986264,0.5,0.4,0.444444,0.290049,0.873816,0.697214


## 5. RG3 학습 — CN7 완료 후 실행

In [6]:
run_dnn('rg3')

rg3 DNN 반복 1, epoch 20
rg3 DNN 반복 1, epoch 40
rg3 DNN 반복 1, epoch 60
rg3 DNN 반복 1, epoch 80
rg3 DNN 반복 1, epoch 100
rg3 DNN 반복 1 완료: 남은 미라벨 32347
rg3 DNN 반복 2, epoch 20
rg3 DNN 반복 2, epoch 40
rg3 DNN 반복 2, epoch 60
rg3 DNN 반복 2, epoch 80
rg3 DNN 반복 2, epoch 100
rg3 DNN 반복 2 완료: 남은 미라벨 29113
rg3 DNN 반복 3, epoch 20
rg3 DNN 반복 3, epoch 40
rg3 DNN 반복 3, epoch 60
rg3 DNN 반복 3, epoch 80
rg3 DNN 반복 3, epoch 100
rg3 DNN 반복 3 완료: 남은 미라벨 26202
rg3 DNN 반복 4, epoch 20
rg3 DNN 반복 4, epoch 40
rg3 DNN 반복 4, epoch 60
rg3 DNN 반복 4, epoch 80
rg3 DNN 반복 4, epoch 100
rg3 DNN 반복 4 완료: 남은 미라벨 23582
rg3 DNN 반복 5, epoch 20
rg3 DNN 반복 5, epoch 40
rg3 DNN 반복 5, epoch 60
rg3 DNN 반복 5, epoch 80
rg3 DNN 반복 5, epoch 100
rg3 DNN 반복 5 완료: 남은 미라벨 21224
rg3 DNN 반복 6, epoch 20
rg3 DNN 반복 6, epoch 40
rg3 DNN 반복 6, epoch 60
rg3 DNN 반복 6, epoch 80
rg3 DNN 반복 6, epoch 100
rg3 DNN 반복 6 완료: 남은 미라벨 19102
rg3 DNN 반복 7, epoch 20
rg3 DNN 반복 7, epoch 40
rg3 DNN 반복 7, epoch 60
rg3 DNN 반복 7, epoch 80
rg3 DNN 반복 7, epoch 100
rg3 DNN 

,track,product,model,iterations,trained,assigned_not_refit,TN,FP,FN,TP,accuracy,precision,recall,F1,AP,ROC_AUC_score,guide_ROC_AUC_hard
0,guide_semi,RG3,DNN,22,32831,393,339,8,8,0,0.95493,0.0,0.0,0.0,0.030099,0.404899,0.488473


## 6. 결과 확인·제출용 ZIP 만들기

DNN 두 제품이 모두 완료되어야 실행된다. ZIP에는 지표·실행 환경·입력 파일 해시·반복 기록만 담으며 원본 데이터와 중간 학습 배열을 넣지 않는다. ZIP과 실행 결과가 저장된 ipynb를 전달하면 최종 프로젝트 정리를 이어갈 수 있다.

In [7]:
dnn=[r for r in rows if r.get('model')=='DNN']
assert {r['product'] for r in dnn}=={'CN7','RG3'},'CN7/RG3 학습을 먼저 완료하세요.'
result=pd.DataFrame(rows)
display(result[['product','model','TP','FN','FP','precision','recall','F1','AP']])
result.to_csv(RESULT_DIR/'metrics.csv',index=False)
import zipfile
archive=RESULT_DIR/'dnn_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for name in ['metrics.csv','cn7_DNN_iterations.csv','rg3_DNN_iterations.csv',
                 'environment.json','input_manifest.csv']:
        z.write(RESULT_DIR/name,arcname=name)
print('두 제품 학습 완료. 결과 ZIP:',archive)
from google.colab import files
files.download(str(archive))


,product,model,TP,FN,FP,precision,recall,F1,AP
0,CN7,DenoisingAE,39,0,2,0.951220,1.0,0.975000,1.000000
1,CN7,SVM,0,5,0,0.000000,0.0,0.000000,0.472582
2,CN7,RandomForest,2,3,2,0.500000,0.4,0.444444,0.475646
3,CN7,GaussianNB,4,1,65,0.057971,0.8,0.108108,0.441209
4,RG3,SVM,0,8,0,0.000000,0.0,0.000000,0.028050
5,RG3,RandomForest,0,8,6,0.000000,0.0,0.000000,0.029681
6,RG3,GaussianNB,8,0,337,0.023188,1.0,0.045326,0.020818
7,CN7,DNN,2,3,2,0.500000,0.4,0.444444,0.290049
8,RG3,DNN,0,8,8,0.000000,0.0,0.000000,0.030099


두 제품 학습 완료. 결과 ZIP: /content/drive/MyDrive/molding_dnn_manual_results/dnn_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 출처와 해석

중소벤처기업부, Korea AI Manufacturing Platform(KAMP), 사출성형기 AI 데이터셋, KAIST(울산과학기술원, ㈜이피엠솔루션즈), 2020.12.14., https://www.kamp-ai.kr/

불량 라벨은 1이다. 가이드의 라벨별 정규화를 사용하는 AE 결과는 운영 검증 성능이 아니다. 준지도 모델의 무작위 평가 결과도 기존 시간순 평가 점수와 직접 개선율을 계산하지 않는다. 최종 반복의 결과를 사용하며, 평가 점수가 좋은 반복을 사후 선택하지 않는다.